# IoT anomaly injector and detector simulator

Run `01_training_fixed.ipynb` first (or `python -m iot_detector.train`) so the models exist.

**How it works.** A stream of device readings goes through three detector layers:
- **Random Forest:** knows the DoS and MITM patterns from the dataset, and also guesses the attack type.
- **Isolation Forest:** flags readings unlike normal traffic.
- **Sensor checks:** range, jump and flatline rules on temperature and humidity.

**Modes**
- *Live:* normal readings from the held-out test set, which the model hasn't seen. Anomalies only appear when you inject them.
- *Replay:* the full test set in its recorded order, real attacks included.

**Climate**, live mode only:
- *Recorded room:* the real temperature and humidity sequence.
- *Synthetic:* a random walk through climates the model hasn't seen, which shows how much it relies on the weather.

**Chart:** orange bands are true anomalies and red dots are detections. An orange band without red dots means a miss;
red dots outside a band are false alarms.

In [1]:
from iot_detector.notebook_ui import SimulatorUI

ui = SimulatorUI(seed=0)
ui.show()

### Things to try
1. Press **Play** and watch the false-alarm rate on normal traffic.
2. Inject **Synthetic DoS** at intensity 0.2, then 1.0. How weak an attack still gets caught?
3. Inject **Synthetic MITM** with only *Random Forest* ticked, then add *Isolation Forest*.
4. Inject **Sensor stuck** with *Sensor checks* off. The models alone never notice.
5. Switch climate to **Synthetic** and watch false alarms rise with no attack at all.

## Scripted scenario

The same simulator without the UI, for repeatable comparisons:

In [2]:
import pandas as pd
from iot_detector.injectors import INJECTORS
from iot_detector.simulator import Simulator

rows = []
for kind in INJECTORS:
    for intensity in (0.3, 1.0):
        s = Simulator(seed=1)
        s.run(30)
        s.inject(kind, intensity)
        h = s.run(INJECTORS[kind].default_length + 10)
        inj = h[h['injected'].notna()]
        ev = s.events_frame().iloc[0]
        rows.append({'injector': INJECTORS[kind].label, 'intensity': intensity,
                     'readings flagged': inj['anomaly'].mean(), 'RF': inj['supervised_flag'].mean(),
                     'IF': inj['novelty_flag'].mean(), 'sensor': inj['sensor_flag'].mean(),
                     'readings to detect': ev['latency']})
pd.DataFrame(rows).style.format({c: '{:.0%}' for c in ['readings flagged', 'RF', 'IF', 'sensor']})

,injector,intensity,readings flagged,RF,IF,sensor,readings to detect
0,Replay DoS,0.300000,100%,100%,90%,0%,0
1,Replay DoS,1.000000,100%,100%,90%,0%,0
2,Replay MITM,0.300000,100%,100%,10%,0%,0
3,Replay MITM,1.000000,100%,100%,10%,0%,0
4,Synthetic DoS,0.300000,100%,100%,95%,0%,0
5,Synthetic DoS,1.000000,100%,100%,100%,0%,0
6,Synthetic MITM,0.300000,100%,85%,100%,0%,0
7,Synthetic MITM,1.000000,100%,100%,100%,30%,0
8,Sensor spike,0.300000,100%,0%,0%,100%,0
9,Sensor spike,1.000000,100%,0%,0%,100%,0


In [3]:
baseline = []
for climate in ('recorded', 'synthetic'):
    s = Simulator(seed=0, climate=climate)
    s.run(400)
    baseline.append({'climate': climate, **s.scoreboard()})
pd.DataFrame(baseline)[['climate', 'readings', 'fp', 'false_alarm_rate']]

,climate,readings,fp,false_alarm_rate
0,recorded,400,8,0.020
1,synthetic,400,42,0.105
